In [46]:
# %pip install python-dateutil

Note: you may need to restart the kernel to use updated packages.


In [47]:
import pandas as pd
import os
import requests
import json
import xml.etree.ElementTree as ET

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import time
import re
import warnings

In [2]:
pd.set_option("display.max_rows", 5000)
warnings.filterwarnings("ignore")

In [53]:
# API 파라미터 설정
base_url = "http://apis.data.go.kr/B551011/DataLabService/locgoRegnVisitrDDList"
mobile_os_lst = ["IOS", "AND", "WIN", "ETC"]
mobile_app = "app"
service_key = os.environ["DATA_GO_KR_SERVICE_KEY"]  # 공공데이터포털 인증키: 환경변수로 전달

start_ymd = "20240101"
end_ymd = "20240131"
limit_ymd = "20250101"

data_list = []
while start_ymd != limit_ymd:
    print("="*50)
    print(f"start_ymd: {start_ymd}, end_ymd: {end_ymd}")
    for mobile_os in mobile_os_lst:
        print("="*50)
        print(f"MobileOS: {mobile_os} 처리 중...")
        for page_no in range(0, 500000):
            # API 호출을 위한 URL 생성
            # 이 때 https를 입력할 경우 SSR 에러 발생 반드시 http를 이용할 것
            url = f"{base_url}?pageNo={page_no}&MobileOS={mobile_os}&MobileApp={mobile_app}&serviceKey={service_key}&startYmd={start_ymd}&endYmd={end_ymd}"

            # API 호출
            response = requests.get(url ,verify= False)
            if response.status_code == 200:
                # 응답 데이터를 XML로 파싱
                root = ET.fromstring(response.text)
                
                # XML을 데이터프레임으로 변환
                for item in root.iter('item'):
                    item_dict = {}
                    for child in item:
                        item_dict[child.tag] = child.text
                    data_list.append(list(item_dict.values()))

                # XML 데이터가 더 이상 없으면 반복문 종료
                if len(root.findall('.//item')) == 0:
                    print(f"페이지 {page_no}에서 데이터가 더 이상 없습니다.")
                    print(f"페이지 {page_no}에서 총 {len(data_list)}건 데이터 추출")
                    time.sleep(2)
                    break
            
            if page_no % 100 == 0:
                time.sleep(5)

    visitor_df = pd.DataFrame(data_list, columns=item_dict.keys())
    visitor_df = visitor_df.drop_duplicates()
    print(visitor_df.shape)
    visitor_df.to_csv(f"../raw_data/visitor_data/visitor_data_{start_ymd}.csv", index=False)

    data_list = []
    start_ymd = datetime.strptime(start_ymd, "%Y%m%d") + relativedelta(months=1)
    end_ymd = start_ymd - timedelta(days=1)

    start_ymd = start_ymd.strftime("%Y%m%d")
    end_ymd = end_ymd.strftime("%Y%m%d")
    time.sleep(2)

In [ ]:
# 전체 일자 데이터 호출 후 병합하고 아래 전처리 수행
import os

csv_files = os.listdir("../raw_data/visitor_data/")
csv_files = [file for file in csv_files if file.endswith(".csv")]

visitor_df = pd.concat([pd.read_csv(f"../raw_data/visitor_data/{file}") for file in csv_files])
print(visitor_df.shape)
visitor_df.head()

In [7]:
def get_sido_code(df):
    # signguCode의 앞 두 자리를 추출해 sidoCode로 생성
    df["sidoCode"] = df["signguCode"].astype(str).str[:2]

    # sidoCode를 시도명으로 매핑하기 위한 사전 예시 (대표적인 것만 일부 기재)
    sido_map = {
        "11": "서울특별시",
        "26": "부산광역시",
        "27": "대구광역시",
        "28": "인천광역시",
        "29": "광주광역시",
        "30": "대전광역시",
        "31": "울산광역시",
        "36": "세종특별자치시",
        "41": "경기도",
        "51": "강원특별자치도",
        "43": "충청북도",
        "44": "충청남도",
        "52": "전북특별자치도",
        "46": "전라남도",
        "47": "경상북도",
        "48": "경상남도",
        "50": "제주특별자치도",
    }

    # sidoCode로 sidoNm 컬럼 생성
    df["sidoNm"] = df["sidoCode"].map(sido_map).fillna("기타")
    return df

In [42]:
# 시도 코드, 이름 컬럼 생성
df = get_sido_code(visitor_df)
df["address"] = df["sidoNm"] + " " + df["signguNm"]

# touDivNm 전처리 및 외국인 데이터 삭제
df["touDivNm"] = df["touDivNm"].apply(lambda x: re.sub(r'\s*\(.*\)', '', x))
df = df[df["touDivNm"] != "외국인"]
df["month"] = pd.to_datetime(df["baseYmd"], format="%Y%m%d").dt.month
df["year"] = pd.to_datetime(df["baseYmd"], format="%Y%m%d").dt.year
df["touNum"] = df["touNum"].astype(float).round(0)
df["touNum"] = df["touNum"].astype(int)

columns = ["address", "baseYmd", "year", "month", "touDivNm", "touNum"]
target_df = df[columns]

target_df.head()

,address,baseYmd,year,month,touDivNm,touNum
0,서울특별시 종로구,20240101,2024,1,현지인,98007
1,서울특별시 종로구,20240101,2024,1,외지인,302838
2,서울특별시 종로구,20240101,2024,1,외국인,32907
3,서울특별시 중구,20240101,2024,1,현지인,76181
4,서울특별시 중구,20240101,2024,1,외지인,283633


In [17]:
gb_add_month = target_df.groupby(["address", "year", "month"]).agg({"touNum": "sum"}).reset_index().sort_values(ascending=False, by="touNum")

gb_add_month["touRatio"] = gb_add_month["touNum"] / gb_add_month["touNum"].sum()
gb_add_month["touRatio"] = gb_add_month["touRatio"].apply(lambda x: round(x, 4)*100)

gb_add_month.head()

,address,year,month,touNum,touRatio
164,서울특별시 종로구,2024,1,3206760,1.67
165,서울특별시 중구,2024,1,2878512,1.50
162,서울특별시 용산구,2024,1,2870496,1.50
159,서울특별시 송파구,2024,1,2622736,1.37
61,경기도 화성시,2024,1,2565120,1.34


In [59]:
gb_add_month.to_csv("../data/visitor_stats_by_city_month.csv", index=False)